In [ ]:
# Cell 1: Setup and imports
from __future__ import annotations

import csv
import importlib.util
import os
import pickle
import random
import subprocess
import sys
import time
from pathlib import Path

import ecole
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import torch
import torch_geometric
from scipy.stats import pearsonr

ROOT = Path.cwd().resolve()
L2B_ECOLE = ROOT / "vast_work" / "learn2branch-ecole"
L2B_ORIGINAL = ROOT / "vast_work" / "learn2branch"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Make Learn2Branch/Ecole imports resolve exactly like the official scripts.
for path in [L2B_ECOLE, L2B_ORIGINAL, ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"root: {ROOT}")
print(f"learn2branch-ecole: {L2B_ECOLE} exists={L2B_ECOLE.exists()}")
print(f"learn2branch: {L2B_ORIGINAL} exists={L2B_ORIGINAL.exists()}")
print(f"ecole: {ecole.__version__}")
print(f"torch: {torch.__version__}")
print(f"torch_geometric: {torch_geometric.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")


In [ ]:
# Cell 2: Generate OOD instances
# The upstream Learn2Branch generator creates full train/valid/test sets if run as a script.
# This cell imports its generator functions and creates only bounded transfer-style sets.

OOD_BASE = L2B_ORIGINAL / "data" / "instances"
TARGET_PER_OOD_CLASS = 100
GEN_SEED = 123

def load_original_generator():
    generator_path = L2B_ORIGINAL / "01_generate_instances.py"
    if not generator_path.exists():
        raise FileNotFoundError(f"Missing generator: {generator_path}")
    spec = importlib.util.spec_from_file_location("l2b_instance_generator", generator_path)
    module = importlib.util.module_from_spec(spec)
    old_path = list(sys.path)
    sys.path.insert(0, str(L2B_ORIGINAL))
    try:
        spec.loader.exec_module(module)  # type: ignore[union-attr]
    finally:
        sys.path[:] = old_path
    return module

generator = load_original_generator()
rng = np.random.RandomState(GEN_SEED)

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)
    return path

def existing_instances(path: Path) -> list[Path]:
    return sorted(path.glob("instance_*.lp")) + sorted(path.glob("instance_*.mps"))

def generate_cauctions(n: int = TARGET_PER_OOD_CLASS):
    out = ensure_dir(OOD_BASE / "cauctions" / "transfer_100_500")
    existing = existing_instances(out)
    for idx in range(len(existing) + 1, n + 1):
        generator.generate_cauctions(rng, str(out / f"instance_{idx}.lp"), n_items=100, n_bids=500, add_item_prob=0.7)
        if idx % 10 == 0 or idx == n:
            print(f"cauctions: generated {idx}/{n}")
    return existing_instances(out)

def generate_facilities(n: int = TARGET_PER_OOD_CLASS):
    out = ensure_dir(OOD_BASE / "facilities" / "transfer_100_100_5")
    existing = existing_instances(out)
    for idx in range(len(existing) + 1, n + 1):
        generator.generate_capacited_facility_location(rng, str(out / f"instance_{idx}.lp"), n_customers=100, n_facilities=100, ratio=5)
        if idx % 10 == 0 or idx == n:
            print(f"facilities: generated {idx}/{n}")
    return existing_instances(out)

def generate_indset(n: int = TARGET_PER_OOD_CLASS):
    out = ensure_dir(OOD_BASE / "indset" / "transfer_500_4")
    existing = existing_instances(out)
    for idx in range(len(existing) + 1, n + 1):
        graph = generator.Graph.barabasi_albert(500, 4, rng)
        generator.generate_indset(graph, str(out / f"instance_{idx}.lp"))
        if idx % 10 == 0 or idx == n:
            print(f"indset: generated {idx}/{n}")
    return existing_instances(out)

def generate_mknapsack(n: int = TARGET_PER_OOD_CLASS):
    # Some Learn2Branch copies do not include an mknapsack generator. If it exists, use it;
    # otherwise reuse any local generated multidimensional knapsack MPS files and skip generation.
    out = ensure_dir(OOD_BASE / "mknapsack" / "transfer_100_6")
    existing = existing_instances(out)
    if len(existing) >= n:
        return existing
    if hasattr(generator, "generate_mknapsack"):
        for idx in range(len(existing) + 1, n + 1):
            generator.generate_mknapsack(rng, str(out / f"instance_{idx}.lp"), number_of_items=100, number_of_constraints=6)
            if idx % 10 == 0 or idx == n:
                print(f"mknapsack: generated {idx}/{n}")
        return existing_instances(out)
    fallback = sorted((ROOT / "generated_instances" / "multidim_knapsack").glob("*.mps"))
    if fallback:
        print(f"mknapsack generator not found; using {len(fallback)} fallback MPS files from generated_instances/multidim_knapsack")
        return fallback
    print("mknapsack generator not found and no fallback files exist; mknapsack will be skipped")
    return []

instance_inventory = {
    "cauctions": generate_cauctions(),
    "facilities": generate_facilities(),
    "indset": generate_indset(),
    "mknapsack": generate_mknapsack(),
}

for name, paths in instance_inventory.items():
    print(f"{name}: {len(paths)} instances")


In [ ]:
# Cell 3: Load model
MODEL_PATH = L2B_ECOLE / "model" / "setcover" / "0" / "train_params.pkl"
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing trained model weights: {MODEL_PATH}")

# Avoid accidentally importing a stale model module from another path.
for key in list(sys.modules):
    if key == "model" or key.startswith("model."):
        del sys.modules[key]

sys.path.insert(0, str(L2B_ECOLE))
from model.model import GNNPolicy

device = "cuda:0" if torch.cuda.is_available() else "cpu"
policy = GNNPolicy().to(device)
policy.load_state_dict(torch.load(MODEL_PATH, map_location=device))
policy.eval()

print(f"loaded GNNPolicy from {L2B_ECOLE / 'model' / 'model.py'}")
print(f"loaded weights from {MODEL_PATH}")
print(f"device: {device}")


In [ ]:
# Cell 4: Load distance metric and build reference set
from milp_distance.distance import extract_normalized_representation, greedy_instance_distance

REFERENCE_SIZE = 40
REFERENCE_SEED = 0
REFERENCE_CACHE = RESULTS_DIR / "reference_reps.pkl"
DISTANCE_CACHE_CSV = RESULTS_DIR / "distance_cache.csv"

train_instance_dir = L2B_ECOLE / "data" / "instances" / "setcover" / "train_500r_1000c_0.05d"
train_instances = sorted(train_instance_dir.glob("instance_*.lp"))
if len(train_instances) < REFERENCE_SIZE:
    raise RuntimeError(f"Need at least {REFERENCE_SIZE} setcover train instances in {train_instance_dir}; found {len(train_instances)}")

rng_ref = np.random.RandomState(REFERENCE_SEED)
reference_paths = [train_instances[i] for i in rng_ref.choice(len(train_instances), size=REFERENCE_SIZE, replace=False)]

if REFERENCE_CACHE.exists():
    with REFERENCE_CACHE.open("rb") as handle:
        payload = pickle.load(handle)
    if payload.get("paths") == [str(p) for p in reference_paths]:
        reference_reps = payload["reps"]
        print(f"loaded {len(reference_reps)} cached reference representations from {REFERENCE_CACHE}")
    else:
        reference_reps = None
else:
    reference_reps = None

if reference_reps is None:
    reference_reps = []
    for i, path in enumerate(reference_paths, start=1):
        print(f"reference {i}/{REFERENCE_SIZE}: {path.name}")
        reference_reps.append(extract_normalized_representation(path))
    with REFERENCE_CACHE.open("wb") as handle:
        pickle.dump({"paths": [str(p) for p in reference_paths], "reps": reference_reps}, handle)

if DISTANCE_CACHE_CSV.exists():
    distance_cache_df = pd.read_csv(DISTANCE_CACHE_CSV)
    distance_cache = dict(zip(distance_cache_df["path"], distance_cache_df["distance"]))
else:
    distance_cache = {}

def save_distance_cache():
    pd.DataFrame(
        [{"path": path, "distance": distance} for path, distance in sorted(distance_cache.items())]
    ).to_csv(DISTANCE_CACHE_CSV, index=False)

def distribution_distance(mps_path) -> float:
    path = str(Path(mps_path).resolve())
    if path in distance_cache:
        return float(distance_cache[path])
    rep = extract_normalized_representation(path)
    distances = [greedy_instance_distance(rep, ref) for ref in reference_reps if ref is not None]
    finite = [d for d in distances if np.isfinite(d)]
    distance = float(np.mean(finite)) if finite else float("inf")
    distance_cache[path] = distance
    if len(distance_cache) % 10 == 0:
        save_distance_cache()
    return distance

print(f"reference reps ready: {len(reference_reps)}")
print(f"distance cache entries: {len(distance_cache)}")


In [ ]:
# Cell 5: Branching evaluation function
SCIP_PARAMS = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 60.0,
    "limits/nodes": 10000,
    "timing/clocktype": 1,
}

def tensorize_node_observation(obs, device):
    return (
        torch.from_numpy(obs.row_features.astype(np.float32)).to(device),
        torch.from_numpy(obs.edge_features.indices.astype(np.int64)).to(device),
        torch.from_numpy(obs.edge_features.values.astype(np.float32)).view(-1, 1).to(device),
        torch.from_numpy(obs.variable_features.astype(np.float32)).to(device),
    )

def solve_vanilla_scip(instance_path, time_limit=60):
    params = dict(SCIP_PARAMS)
    params["limits/time"] = float(time_limit)
    env = ecole.environment.Configuring(scip_params=params)
    wall_start = time.perf_counter()
    env.reset(str(instance_path))
    _, _, _, _, _ = env.step({})
    walltime = time.perf_counter() - wall_start
    model = env.model.as_pyscipopt()
    return {
        "scip_nodes": int(model.getNNodes()),
        "scip_lps": int(model.getNLPs()),
        "scip_time": float(model.getSolvingTime()),
        "scip_walltime": float(walltime),
        "scip_status": str(model.getStatus()),
        "scip_gap": float(model.getGap()),
    }

def solve_ml_branching(instance_path, policy, device, time_limit=60):
    params = dict(SCIP_PARAMS)
    params["limits/time"] = float(time_limit)
    env = ecole.environment.Branching(
        observation_function=ecole.observation.NodeBipartite(),
        scip_params=params,
        pseudo_candidates=False,
    )
    env.seed(0)
    torch.manual_seed(0)
    wall_start = time.perf_counter()

    obs, action_set, _, done, _ = env.reset(str(instance_path))
    while not done:
        if action_set is None or len(action_set) == 0:
            break
        with torch.no_grad():
            tensors = tensorize_node_observation(obs, device)
            logits = policy(*tensors)
            action_tensor_index = torch.as_tensor(action_set.astype(np.int64), device=device)
            selected = int(torch.argmax(logits[action_tensor_index]).item())
            action = int(action_set[selected])
        obs, action_set, _, done, _ = env.step(action)

    walltime = time.perf_counter() - wall_start
    model = env.model.as_pyscipopt()
    return {
        "ml_nodes": int(model.getNNodes()),
        "ml_lps": int(model.getNLPs()),
        "ml_time": float(model.getSolvingTime()),
        "ml_walltime": float(walltime),
        "ml_status": str(model.getStatus()),
        "ml_gap": float(model.getGap()),
    }

def evaluate_instance(mps_path, policy, device, time_limit=60) -> dict | None:
    try:
        path = Path(mps_path)
        scip = solve_vanilla_scip(path, time_limit=time_limit)
        ml = solve_ml_branching(path, policy, device, time_limit=time_limit)
        scip_nodes = max(1, int(scip["scip_nodes"]))
        ml_nodes = int(ml["ml_nodes"])
        return {
            **scip,
            **ml,
            "degradation": float(ml_nodes / scip_nodes),
        }
    except Exception as exc:
        print(f"[SKIP] {mps_path}: {type(exc).__name__}: {exc}")
        return None

print("evaluation functions ready")


In [ ]:
# Cell 6: Run experiment
RAW_RESULTS_CSV = RESULTS_DIR / "raw_results.csv"
N_PER_CLASS = 50
TIME_LIMIT = 60

def first_n(paths, n=N_PER_CLASS):
    return [Path(p) for p in sorted(paths)[:n]]

setcover_valid_dir = L2B_ECOLE / "data" / "instances" / "setcover" / "valid_500r_1000c_0.05d"

# Prefer the generated transfer sets. The mknapsack entry may come from a fallback MPS folder.
test_sets = {
    "setcover_valid": first_n(setcover_valid_dir.glob("instance_*.lp")),
    "cauctions": first_n((OOD_BASE / "cauctions" / "transfer_100_500").glob("instance_*.lp")),
    "facilities": first_n((OOD_BASE / "facilities" / "transfer_100_100_5").glob("instance_*.lp")),
    "indset": first_n((OOD_BASE / "indset" / "transfer_500_4").glob("instance_*.lp")),
    "mknapsack": first_n(list((OOD_BASE / "mknapsack" / "transfer_100_6").glob("instance_*.lp")) + list((ROOT / "generated_instances" / "multidim_knapsack").glob("*.mps"))),
}

test_sets = {name: paths for name, paths in test_sets.items() if paths}
for name, paths in test_sets.items():
    print(f"{name}: {len(paths)} instances")

if RAW_RESULTS_CSV.exists():
    results_df = pd.read_csv(RAW_RESULTS_CSV)
    completed = set(results_df["path"].astype(str))
    rows = results_df.to_dict("records")
    print(f"loaded {len(rows)} existing results from {RAW_RESULTS_CSV}")
else:
    completed = set()
    rows = []

for class_name, paths in test_sets.items():
    for idx, path in enumerate(paths, start=1):
        resolved = str(path.resolve())
        if resolved in completed:
            continue
        distance = distribution_distance(path)
        metrics = evaluate_instance(path, policy, device, time_limit=TIME_LIMIT)
        if metrics is None:
            continue
        row = {
            "class": class_name,
            "path": resolved,
            "distance": distance,
            "scip_nodes": metrics["scip_nodes"],
            "ml_nodes": metrics["ml_nodes"],
            "degradation": metrics["degradation"],
            "scip_lps": metrics["scip_lps"],
            "ml_lps": metrics["ml_lps"],
            "scip_time": metrics["scip_time"],
            "ml_time": metrics["ml_time"],
            "scip_walltime": metrics["scip_walltime"],
            "ml_walltime": metrics["ml_walltime"],
            "scip_status": metrics["scip_status"],
            "ml_status": metrics["ml_status"],
            "scip_gap": metrics["scip_gap"],
            "ml_gap": metrics["ml_gap"],
        }
        rows.append(row)
        completed.add(resolved)
        pd.DataFrame(rows).to_csv(RAW_RESULTS_CSV, index=False)
        save_distance_cache()
        if idx % 10 == 0 or idx == 1:
            print(f"{class_name} {idx}/{len(paths)} distance={distance:.4f} degradation={metrics['degradation']:.3f}")

results_df = pd.DataFrame(rows)
results_df.to_csv(RAW_RESULTS_CSV, index=False)
save_distance_cache()
print(f"saved {len(results_df)} rows to {RAW_RESULTS_CSV}")
results_df.head()


In [ ]:
# Cell 7: Analysis and plots
results_df = pd.read_csv(RAW_RESULTS_CSV)
results_df = results_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["distance", "degradation"])
if results_df.empty:
    raise RuntimeError("No usable results found. Run Cell 6 first.")

r, p_value = pearsonr(results_df["distance"], results_df["degradation"])
print(f"Pearson r(distance, degradation): {r:.4f}, p={p_value:.4g}")

# Plot 1: scatter with regression line.
plt.figure(figsize=(9, 6))
classes = sorted(results_df["class"].unique())
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    plt.scatter(subset["distance"], subset["degradation"], label=class_name, alpha=0.75)

x = results_df["distance"].to_numpy(dtype=float)
y = results_df["degradation"].to_numpy(dtype=float)
coef = np.polyfit(x, y, deg=1)
x_line = np.linspace(np.nanmin(x), np.nanmax(x), 200)
y_line = coef[0] * x_line + coef[1]
plt.plot(x_line, y_line, color="black", linewidth=2, label="linear fit")
plt.axhline(1.0, color="red", linestyle="--", linewidth=1.5)
plt.xlabel("Mean Maudet distance to setcover training reference")
plt.ylabel("Degradation ratio: ML nodes / vanilla SCIP nodes")
plt.title("MILP Distance vs ML Branching Degradation")
plt.text(0.02, 0.98, f"Pearson r={r:.3f}\np={p_value:.2g}", transform=plt.gca().transAxes, va="top")
plt.legend()
plt.tight_layout()
scatter_path = RESULTS_DIR / "scatter.png"
plt.savefig(scatter_path, dpi=200)
plt.show()
print(f"saved {scatter_path}")

# Plot 2: box plot ordered by median distance.
order = results_df.groupby("class")["distance"].median().sort_values().index.tolist()
box_data = [results_df.loc[results_df["class"] == cls, "degradation"].to_numpy() for cls in order]
plt.figure(figsize=(9, 6))
plt.boxplot(box_data, labels=order, showmeans=True)
plt.axhline(1.0, color="red", linestyle="--", linewidth=1.5)
plt.xlabel("Problem class ordered by median distance")
plt.ylabel("Degradation ratio: ML nodes / vanilla SCIP nodes")
plt.title("Degradation by Problem Class")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
boxplot_path = RESULTS_DIR / "boxplot.png"
plt.savefig(boxplot_path, dpi=200)
plt.show()
print(f"saved {boxplot_path}")

summary = results_df.groupby("class").agg(
    n=("degradation", "size"),
    median_distance=("distance", "median"),
    median_degradation=("degradation", "median"),
    mean_degradation=("degradation", "mean"),
    frac_degraded=("degradation", lambda s: float(np.mean(s > 1.0))),
).sort_values("median_distance")
print("\nSummary by class:")
display(summary)

# Best threshold separating degraded (>1) from not degraded by balanced accuracy.
thresholds = np.unique(results_df["distance"].to_numpy(dtype=float))
y_true = (results_df["degradation"].to_numpy(dtype=float) > 1.0)
best = None
for threshold in thresholds:
    pred = results_df["distance"].to_numpy(dtype=float) > threshold
    tp = np.sum(pred & y_true)
    tn = np.sum(~pred & ~y_true)
    fp = np.sum(pred & ~y_true)
    fn = np.sum(~pred & y_true)
    tpr = tp / max(1, tp + fn)
    tnr = tn / max(1, tn + fp)
    balanced_acc = 0.5 * (tpr + tnr)
    if best is None or balanced_acc > best["balanced_accuracy"]:
        best = {
            "threshold": float(threshold),
            "balanced_accuracy": float(balanced_acc),
            "tpr": float(tpr),
            "tnr": float(tnr),
            "above_threshold_count": int(np.sum(pred)),
            "above_threshold_degraded_pct": float(100 * np.mean(y_true[pred])) if np.any(pred) else float("nan"),
        }

print("\nBest distance threshold for degradation > 1:")
for key, value in best.items():
    print(f"{key}: {value}")
